# How to publish your code outputs to the SDL — step-by-step guide (EDANYA)

This guide is for **EDANYA group members** who want to publish the outputs of a HySEA code (Tsunami /
Landslide / Multilayer) to the **SDL** (*Simulation Data Lake*, CINECA / Geo-INQUIRE). It walks you from
**how to lay out your folders** to **managing codes and people/projects/organizations**, then
**generating the manifest** and **publishing** — using the three tools under the **HySEALab ▸ SDL
Management** menu.

**The SDL data model**

```
Experiment  (your run / campaign)
└── Version (e.g. 1.0.0)
    ├── scenario/   ← the topo-bathymetry (uploaded ONCE, shared)
    └── simulation/ ← one entry per parameter file (+ its outputs)
```

**The whole process in 4 moves**

1. **Lay out your project folder** correctly (Section 1) — `parfiles/ bathy/ sim/`.
2. **Register your code** if needed → *HySEALab ▸ SDL Management ▸ HySEA Codes registry manager* (Section 2).
3. **Make sure authors / projects / organizations exist** → *Catalogue manager* (Section 3).
4. **Generate the manifest & publish** → *Generate manifest & publish* (Section 4).

> You only need your **SDL account** (username + password); you type it at run time and nothing is
> stored. Test on the **dev** server first, then switch to **prod** for the real publication.


## 1. Folder layout & naming (the requirements)

Put everything for one experiment in a single **project folder** with **exactly these three
subfolders**:

```
my_experiment/
├── parfiles/   one or more Tsunami-HySEA parameter files (*.txt)  -> each becomes a SIMULATION
├── bathy/      the topo-bathymetry grid(s) (*.grd / *.nc)         -> the SCENARIO (uploaded once)
└── sim/        the model outputs of each run                      -> matched to their parfile
```

**Naming rules that matter**

- **Each `.txt` in `parfiles/` becomes one SDL *simulation*.** Its name and its **NetCDF output prefix**
  are read from inside the parameter file.
- **Outputs in `sim/` must start with that prefix** so the publisher can match them to the right
  simulation. Example: `parfiles/AT005_v1.txt` (prefix `AT005_v1_l0_1`) → `sim/AT005_v1_l0_1.nc`
  (`.bin` / `.deb` also fine).
- **One shared `bathy/` grid + several parfiles = a parameter sweep**: 1 experiment, 1 scenario, N
  simulations. (See the examples `../ejemplo_SDL` = 1 sim, `../ejemplo_5sims` = 5 sims.)
- You do **not** create the bathymetry figures — the publisher generates them automatically.

Run the optional check below to validate a folder before you publish.

### Setup (only needed for the optional folder check)

In [ ]:
# --- Setup: load the companion library sdl_lib.py and check dependencies -----
import os, sys, importlib, importlib.util
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display, clear_output

# Locate sdl_lib.py: it lives either in a sibling 01_sdl_publish/ folder (repo
# layout) or directly in the parent folder (server ~/SDL layout). Search both.
_cands = [os.path.join(os.getcwd(), "..", "01_sdl_publish"),
          os.path.join(os.getcwd(), ".."),
          os.getcwd(),
          os.path.expanduser("~/HySEALab/Notebooks/01_sdl_publish"),  # HySEALab menu: shared lib
          os.path.expanduser("~/HySEALab/Notebooks"),                 # shared (next to map viewer)
          os.path.expanduser("~/SDL")]                                # personal working copy
LIB_DIR = next((os.path.abspath(d) for d in _cands
                if os.path.exists(os.path.join(d, "sdl_lib.py"))), None)
if LIB_DIR is None:
    raise FileNotFoundError("sdl_lib.py not found in ../01_sdl_publish, .. or .")
sys.path.insert(0, LIB_DIR)

REQUIRED = {
    "requests": "requests", "requests_toolbelt": "requests-toolbelt",
    "rasterio": "rasterio", "cmocean": "cmocean",
    "matplotlib": "matplotlib", "numpy": "numpy", "tqdm": "tqdm",
    "yaml": "pyyaml", "jsonschema": "jsonschema",
}
_missing = [pip for mod, pip in REQUIRED.items() if importlib.util.find_spec(mod) is None]

_dep_out = widgets.Output()
def _install_missing(_):
    import subprocess
    with _dep_out:
        clear_output()
        print("Installing into the user site:", " ".join(_missing))
        subprocess.run([sys.executable, "-m", "pip", "install", "--user", *_missing])
        print("Done. Restart the kernel and re-run this cell.")
_btn_install = widgets.Button(description="Install missing deps", button_style="warning")
_btn_install.on_click(_install_missing)

if _missing:
    print("Missing dependencies:", ", ".join(_missing))
    display(_btn_install, _dep_out)
else:
    print("All dependencies available.")

import sdl_lib
importlib.reload(sdl_lib)
print("sdl_lib loaded from:", LIB_DIR)

# Shared state for this notebook (login token lives only in memory).
S = {"base_url": "https://sdl-dev.hpc.cineca.it", "token": None, "headers": None}
print("Setup ready.")


In [ ]:
# --- (Optional) Check a project folder before publishing ---------------------
chk = widgets.Text(description="Project:", layout=widgets.Layout(width="700px"),
                   value=os.path.abspath(os.path.join(os.getcwd(), "..", "ejemplo_5sims")))
chk_btn = widgets.Button(description="Check folder", button_style="info")
chk_out = widgets.Output()

def _check(_):
    with chk_out:
        clear_output()
        root = chk.value.strip()
        if not os.path.isdir(root):
            print("Not a folder:", root); return
        ok = True
        for d in ("parfiles", "bathy", "sim"):
            present = os.path.isdir(os.path.join(root, d))
            print(("  OK   " if present else "  MISSING  ") + d + "/")
            ok = ok and present
        if not ok:
            print("\nThe project folder MUST contain parfiles/, bathy/ and sim/."); return
        grids = [f for f in os.listdir(os.path.join(root, "bathy"))
                 if f.lower().endswith((".grd", ".nc"))]
        print("\nScenario bathymetry (uploaded once, shared by all simulations):",
              ", ".join(grids) or "(NONE - at least one .grd/.nc is required)")
        print("\nSimulations detected (one per parameter file in parfiles/):")
        any_par = False
        for f in sorted(os.listdir(os.path.join(root, "parfiles"))):
            if not f.lower().endswith((".txt", ".par")):
                continue
            any_par = True
            meta = sdl_lib.scan_hysea_parfile(os.path.join(root, "parfiles", f))
            name = meta.get("id_nombre", "?")
            prefixes = [v for k, v in meta.items() if k.startswith("id_NetCDF_prefix")]
            matches = [g for r, _, fl in os.walk(os.path.join(root, "sim"))
                       for g in fl if any(g.startswith(px) for px in prefixes)]
            flag = "OK" if matches else "NO MATCHING OUTPUT in sim/ !"
            print(f"  - {f}: name='{name}', output prefix(es)={prefixes}")
            print(f"      outputs: {matches if matches else '(none)'}   [{flag}]")
        if not any_par:
            print("  (no .txt parameter file found in parfiles/)")
        print("\nIf every simulation shows OK, the folder is ready for "
              "'Generate manifest & publish'.")
chk_btn.on_click(_check)
display(widgets.VBox([chk, chk_btn, chk_out]))


## 2. Register your code (if it is not there yet)

Open **HySEALab ▸ SDL Management ▸ HySEA Codes registry manager**. Codes live in an editable
`sdl_codes.yml`; each has a **display name**, **keywords** and a **theme**. Today the registry has
`tsunami_hysea`, `landslide_hysea`, `multilayer_hysea`.

- If you publish with one of those, there is **nothing to do** — its keywords are applied automatically.
- If you use a **new code**, add it there first (id, display name, keywords). It will then appear in the
  *code selector* of the publish form, and its keywords will be filled in for you.

## 3. Make sure authors / projects / organizations exist

A *new* experiment needs **at least one author**, and may reference **projects** and **organizations**.
The manifest links them by **numeric id**. Open **HySEALab ▸ SDL Management ▸ Catalogue manager** to:

- **Search** the three catalogues (persons / organizations / projects) and note the **ids** you need.
- **Create** an entry if someone/something is missing (the form adapts: persons need given/family/email;
  organizations a *legalName*; projects a *title*).

> Useful ids (dev): UMA = organization **41**, Geo-INQUIRE = project **3**. Author ids are personal —
> look yours up. *Ids differ between dev and prod.*

The *Generate manifest & publish* form also has the same search cells built in, so you can do this there
too.

## 4. Generate the manifest & publish

Open **HySEALab ▸ SDL Management ▸ Generate manifest & publish** and:

1. **Pick your code** from the selector → its **keywords auto-fill** (editable).
2. Fill the **required** fields (marked `*`): **project folder**, **code version**, **title**,
   **author ids** (≥1).
3. Add the **optional** fields as needed: organization ids, project ids, description, **visibility**
   (`private`/`public`) + **license** (only if public), **requestDOI**.
4. **1) Build manifest** → review the generated `sdl_manifest.yml` (technical fields — simulation names,
   grids, output prefixes, spatial coverage — are derived automatically).
5. **2) Publish to SDL** → it creates the experiment/version, uploads the bathymetry once, registers each
   simulation with its outputs, writes the README and sets visibility/license.

**Choices to keep in mind**

- **dev vs prod**: the publish targets the **server you logged into**. Rehearse on **dev**.
- **private vs public**: public requires a **Creative-Commons license** (e.g. `cc-by-4.0`).
- **DOI**: tick *requestDOI* only for a real, citable release (not for tests).
- **Adding more runs later**: you can **append** simulations to an existing experiment *without
  re-uploading the bathymetry* (the `ejemplo_5sims_append` pattern / append manifest).

## Troubleshooting

- **A simulation shows "NO MATCHING OUTPUT"** in the check → the file in `sim/` does not start with the
  parameter file's NetCDF prefix. Rename the output (or fix the prefix in the parfile).
- **"A new experiment needs at least one author"** → set at least one author id in the form.
- **Spatial coverage not computed** → the grid is in a local/metric projection outside lon/lat range;
  it is skipped (the experiment still publishes).
- **Version rejected** → use a valid semantic version such as `1.0.0`.
- **Login fails** → check the server (dev/prod) matches your account and credentials.

## Quick checklist

- [ ] Folder has `parfiles/`, `bathy/`, `sim/`; outputs match each parfile's prefix (run the check).
- [ ] Your code is in the *HySEA Codes registry* (or you added it).
- [ ] Author/project/organization ids known (via *Catalogue manager*).
- [ ] *Generate manifest & publish*: required fields filled → Build → review → Publish (dev first).
